In [1]:
from p4p.client.thread import Context
import p4p
from pprint import pprint
import numpy as np
from pathlib import Path
import socket
import threading
import time

In [2]:
print(Context.providers())
ctxt = Context('pva', conf={
    'EPICS_PVA_AUTO_ADDR_LIST': '0',
    'EPICS_PVA_ADDR_LIST': '127.0.0.1'
})
pprint(vars(ctxt))

['pva']
{'_Q': None,
 '_T': None,
 '_channel_lock': <unlocked _thread.lock object at 0x7fb1c83d6090>,
 '_ctxt': <p4p.client.raw._ClientProvider object at 0x7fb1c83d5910>,
 '_nt': ClientUnwrapper({'epics:nt/NTScalar:1.0': <class 'p4p.nt.scalar.NTScalar'>, 'epics:nt/NTScalarArray:1.0': <class 'p4p.nt.scalar.NTScalar'>, 'epics:nt/NTEnum:1.0': <class 'p4p.nt.enum.NTEnum'>, 'epics:nt/NTNDArray:1.0': <class 'p4p.nt.ndarray.NTNDArray'>}),
 'conf': <bound method ClientProvider.conf of <p4p.client.raw._ClientProvider object at 0x7fb1c83d5910>>,
 'hurryUp': <bound method ClientProvider.hurryUp of <p4p.client.raw._ClientProvider object at 0x7fb1c83d5910>>,
 'name': 'pva'}


In [3]:
val = ctxt.get('tpx:pipe:path')
print(val.raw.value)
ctxt.put('tpx:pipe:path',str(val.raw.value))
val = ctxt.get('tpx:pipe:path')
pprint(val.raw.value)
ctxt.put('tpx:pipe:sid',0)
val = ctxt.get('tpx:pipe:sid')
pprint(val.raw.value)

/home/mtaka/Documents/tpx_pipeline/data
'/home/mtaka/Documents/tpx_pipeline/data'
0.0


In [4]:
path = Path.cwd().parent / "tpx_data"/ "raw"
files = []
for f in path.iterdir():
    if "tpx" in f.suffix:
        files.append(f)
for f in path.iterdir():
    if "tpx" in f.suffix:
        files.append(f)
for f in path.iterdir():
    if "tpx" in f.suffix:
        files.append(f)
pprint(files)

[PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000000.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000001.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000002.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000003.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000004.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000005.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000006.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000007.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000008.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000009.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000010.tpx3'),
 PosixPath('/home/mtaka/Documents/code_tests/tpx_data/raw/rawAO1_000011.tpx3'),
 PosixPath('/home/mtaka/Documents/code_t

In [5]:
HOST = "localhost"
SERVAL = 8088
def server():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        print("[server]\t starting up")
        sock.bind((HOST,SERVAL))
        sock.listen()
        conn, addr = sock.accept()
        print("[server]\t connected, sending files")
        with conn:
            for f in files:
                a = np.fromfile(f, dtype="<u8")
                conn.sendall(a)

        print("[server]\t finished sending files, closing")

In [6]:
res =[]
def cb(V):
    print("new file posted: ", V)
    res.append(V)

ctxt.monitor('tpx:pipe:file',cb)

new file posted:  Wed Dec 31 19:00:00 1969 'false'


In [ ]:
t = threading.Thread(target=server)
t.start()
time.sleep(1)
ctxt.put("tpx:pipe:fire",True)

[server]	 starting up
[server]	 connected, sending files


new file posted:  Wed Dec 31 19:00:00 1969 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_1.parquet'
[server]	 finished sending files, closing
new file posted:  Wed Dec 31 19:00:00 1969 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_5.parquet'
new file posted:  Wed Dec 31 19:00:00 1969 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_0.parquet'
new file posted:  Wed Dec 31 19:00:00 1969 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_3.parquet'
new file posted:  Wed Dec 31 19:00:00 1969 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_2.parquet'
new file posted:  Wed Dec 31 19:00:00 1969 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_4.parquet'
new file posted:  Wed Dec 31 19:00:00 1969 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_6.parquet'
new file posted:  Wed Dec 31 19:00:00 1969 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_7.parquet'
new file posted:  Wed Dec 31 19:00:00 1969 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_10.parquet'
new file posted:  Wed

In [8]:
pprint(res)

['false',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_1.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_5.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_0.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_3.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_2.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_4.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_6.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_7.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_10.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_11.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_8.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_9.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_12.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_17.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_14.parquet',
 '/home/mtaka/Documents/tpx_pipeline/dat

In [11]:

val = ctxt.get('tpx:pipe:fire')
pprint(val.raw.value)
val = ctxt.get('tpx:pipe:file')
pprint(val.raw.value)
val = ctxt.get('tpx:pipe:files')
pprint(sorted(val.raw.value,key=lambda x:int(Path(x).stem.split("_")[-1])))

False
'/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_16.parquet'
['/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_0.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_1.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_2.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_3.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_4.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_5.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_6.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_7.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_8.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_9.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_10.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_11.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_12.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff_0_0_13.parquet',
 '/home/mtaka/Documents/tpx_pipeline/data/buff

In [15]:
import ophyd
print(dir(ophyd.status))

['AndStatus', 'DeviceStatus', 'InvalidState', 'LoggerAdapter', 'MoveStatus', 'StableSubscriptionStatus', 'Status', 'StatusBase', 'StatusTimeoutError', 'SubscriptionStatus', 'UnknownStatusFailure', 'UseNewProperty', 'WaitTimeoutError', '_TRACE_PREFIX', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_set_trace_attributes', 'adapt_old_callback_signature', 'deque', 'json', 'logger', 'np', 'partial', 'threading', 'time', 'trace', 'tracer', 'wait', 'warn']


In [ ]:
from ophyd import Device, Component, DeviceStatus, EpicsSignal, EpicsSignalRO
from ophyd.status import SubscriptionStatus


class tpx3_pipe(Device):
    sid = Component(EpicsSignal,"sid")
    scan = Component(EpicsSignal,"scan")
    path = Component(EpicsSignal,"path")
    active = Component(EpicsSignal,"active")
    fire = Component(EpicsSignal,"fire")
    file_stream = Component(EpicsSignalRO,"file")

    def trigger(self):
        def await_revert(self,was_active,active,**kwargs):
            return (was_active) and (not active)
        
        status = SubscriptionStatus(self.fire,await_revert,run=False)
        self.fire.set(True)
        return status

    def read(self):
        return 

pipeline = tpx3_pipe("tpx:pipe:",name="tpx_pipe")


In [12]:
pprint(dir(pipeline))

['OphydAttrList',
 'SUB_ACQ_DONE',
 '_Device__any_instantiated',
 '_Device__default_connection_timeout',
 '_Device__set_kinds_according_to_list',
 '_OphydObject__any_instantiated',
 '_OphydObject__instantiation_callbacks',
 '_OphydObject__register_instance',
 '__annotations__',
 '__class__',
 '__copy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getnewargs_ex__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_args_cache',
 '_attr_name',
 '_callbacks',
 '_cb_count',
 '_child_name_separator',
 '_cid_to_event_mapping',
 '_component_kinds',
 '_connection_timeout',
 '_default_configuration_attrs',
 '_default_read_attrs',
 '_default_sub',
 '_des

In [13]:
pipeline.summary()

ConnectionTimeoutError: Failed to connect to tpx:pipe:sid within 1.00 sec